# NLP handbook — Part 1: classical NLP (tokenization, TF-IDF, classification, topics, n-gram LMs, embeddings, POS/NER)
Runs on CPU in Colab (~10-20 min total). Each section corresponds to a chapter in `09-nlp/`. Numbers in the chapters came from running these exact cells.

In [ ]:
!pip -q install nltk scikit-learn scipy sklearn-crfsuite seqeval
import nltk
for p in ['punkt','punkt_tab','treebank','movie_reviews','conll2002','stopwords','wordnet','averaged_perceptron_tagger_eng','universal_tagset','brown']: nltk.download(p, quiet=True)

## 1. Tokenization, normalization, Zipf/Heaps, TF-IDF, sentiment classification ablations (chapters N.1-N.3)

In [ ]:
"""NLP basics (executed): tokenization, normalization, Zipf/Heaps, BoW/TF-IDF, classification ablations, LSA, topic models, n-gram LMs."""
import re, math, collections, numpy as np, warnings; warnings.filterwarnings("ignore")
import nltk
from nltk.corpus import movie_reviews, brown, stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, HashingVectorizer, TfidfTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold

print("== tokenization contrasts ==")
s = "Dr. Smith didn't pay $3.50 for U.S. e-mail; it's well-known... #NLP @user https://x.co"
print("whitespace:", s.split())
print("regex \\w+ :", re.findall(r"\w+", s))
print("nltk word_tokenize:", nltk.word_tokenize(s))
print("sentences:", nltk.sent_tokenize("Dr. Smith arrived at 5 p.m. He left. Mr. Lee stayed."))
ps, wl = PorterStemmer(), WordNetLemmatizer()
for w in ["running", "studies", "better", "happily", "universities", "was"]:
    print(f"{w:12s} stem={ps.stem(w):10s} lemma(v)={wl.lemmatize(w, 'v'):8s} lemma(n)={wl.lemmatize(w):12s}")

print("\n== Zipf and Heaps on Brown corpus ==")
toks = [w.lower() for w in brown.words() if w.isalpha()]; cnt = collections.Counter(toks); print("tokens", len(toks), "types", len(cnt))
for r in [1, 10, 100, 1000, 10000]:
    w, f = cnt.most_common(r)[-1]; print(f"rank {r:5d} word {w:12s} freq {f:6d} rank*freq {r*f}")
hap = sum(1 for f in cnt.values() if f == 1); print(f"hapax legomena: {hap} ({100*hap/len(cnt):.1f}% of types)")
for n in [10000, 100000, 500000]: print(f"first {n} tokens -> {len(set(toks[:n]))} types")

print("\n== TF-IDF worked example ==")
docs = ["the cat sat on the mat", "the dog sat on the log", "cats and dogs"]
cv = CountVectorizer(); C = cv.fit_transform(docs); print(cv.get_feature_names_out().tolist()); print(C.toarray())
N = 3; df = (C.toarray() > 0).sum(0); idf_sk = np.log((1 + N) / (1 + df)) + 1
tf = TfidfVectorizer(norm=None).fit(docs); print("idf sklearn:", dict(zip(cv.get_feature_names_out(), tf.idf_.round(3))))
print("manual: idf('cat') = ln((1+3)/(1+1))+1 =", round(math.log(4 / 2) + 1, 3), "| idf('the') = ln(4/3)+1 =", round(math.log(4 / 3) + 1, 3))

print("\n== sentiment classification ablations on movie_reviews (2000 docs, 5-fold CV acc) ==")
texts = [movie_reviews.raw(f) for f in movie_reviews.fileids()]; y = np.array([1 if f.startswith("pos") else 0 for f in movie_reviews.fileids()])
cv5 = StratifiedKFold(5, shuffle=True, random_state=0)
def acc(vec, clf): return cross_val_score(make_pipeline(vec, clf), texts, y, cv=cv5, n_jobs=1).mean()
stem_an = lambda d: [ps.stem(t) for t in re.findall(r"[a-z']+", d.lower())]
rows = [("majority", CountVectorizer(), None),
 ("BoW unigram + NB", CountVectorizer(), MultinomialNB()),
 ("BoW binary + NB", CountVectorizer(binary=True), MultinomialNB()),
 ("BoW unigram + LR", CountVectorizer(), LogisticRegression(max_iter=3000)),
 ("TF-IDF unigram + LR", TfidfVectorizer(sublinear_tf=True), LogisticRegression(max_iter=3000)),
 ("TF-IDF uni+bigram + LR", TfidfVectorizer(sublinear_tf=True, ngram_range=(1, 2), min_df=2), LogisticRegression(max_iter=3000)),
 ("TF-IDF uni+bigram + LinearSVC", TfidfVectorizer(sublinear_tf=True, ngram_range=(1, 2), min_df=2), LinearSVC()),
 ("TF-IDF unigram, stopwords removed", TfidfVectorizer(sublinear_tf=True, stop_words="english"), LogisticRegression(max_iter=3000)),
 ("TF-IDF unigram, Porter stemming", TfidfVectorizer(sublinear_tf=True, analyzer=stem_an), LogisticRegression(max_iter=3000)),
 ("Hashing 2^16 (no idf) + LR, C=1", HashingVectorizer(n_features=2 ** 16, ngram_range=(1, 2), alternate_sign=False), LogisticRegression(max_iter=3000)),
 ("Hashing 2^16 + tf-idf weighting + LR", make_pipeline(HashingVectorizer(n_features=2 ** 16, ngram_range=(1, 2), alternate_sign=False, norm=None), TfidfTransformer(sublinear_tf=True)), LogisticRegression(max_iter=3000))]
for n, v, c in rows:
    if c is None: print(f"{n:34s} {max(y.mean(), 1 - y.mean()):.3f}"); continue
    print(f"{n:34s} {acc(v, c):.3f}")

## 2. Topic models and n-gram language models (chapters N.4, N.6)

In [ ]:
"""Topic models (LSA/NMF/LDA) and n-gram language models with smoothing, evaluated by perplexity (executed)."""
import math, collections, numpy as np, warnings; warnings.filterwarnings("ignore")
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation, NMF, TruncatedSVD
from nltk.corpus import brown

cats = ["sci.space", "rec.autos", "talk.politics.misc", "comp.graphics", "rec.sport.hockey", "sci.med"]
d = fetch_20newsgroups(subset="train", categories=cats, remove=("headers", "footers", "quotes")); texts = d.data
def top(components, names, n=8): return [" ".join(names[i] for i in c.argsort()[::-1][:n]) for c in components]
tf = TfidfVectorizer(stop_words="english", max_df=.5, min_df=5); X = tf.fit_transform(texts)
print("== NMF topics (TF-IDF) =="); [print(f" {i}: {t}") for i, t in enumerate(top(NMF(6, init="nndsvd", random_state=0, max_iter=400).fit(X).components_, tf.get_feature_names_out()))]
print("== LDA topics (counts) ==")
cv = CountVectorizer(stop_words="english", max_df=.5, min_df=5); C = cv.fit_transform(texts)
lda = LatentDirichletAllocation(6, learning_method="batch", max_iter=25, random_state=0).fit(C)
[print(f" {i}: {t}") for i, t in enumerate(top(lda.components_, cv.get_feature_names_out()))]
purity = []
th = lda.transform(C).argmax(1)
tab = np.zeros((6, 6), int)
for t, y in zip(th, d.target): tab[t, y] += 1
print("LDA topic->true category purity:", round(tab.max(1).sum() / tab.sum(), 3), "(random ~", round(1 / 6, 2), ")")
print("LDA perplexity (held-in) for K=2,6,12:", [round(LatentDirichletAllocation(k, max_iter=15, random_state=0).fit(C).perplexity(C), 0) for k in (2, 6, 12)], "(lower is better but held-in perplexity favors big K)")
svd = TruncatedSVD(6, random_state=0).fit(X); print("LSA explained variance ratio:", svd.explained_variance_ratio_.round(3))

print("\n== n-gram language models on Brown (news) ==")
sents = [[w.lower() for w in s] for s in brown.sents(categories="news")]; split = int(len(sents) * .9); tr, te = sents[:split], sents[split:]
cnt1 = collections.Counter(w for s in tr for w in s); thr = 2
vocab = {w for w, c in cnt1.items() if c >= thr} | {"<s>", "</s>", "<unk>"}
prep = lambda ss: [["<s>"] + [w if w in vocab else "<unk>" for w in s] + ["</s>"] for s in ss]
trp, tep = prep(tr), prep(te); V = len(vocab)
u = collections.Counter(w for s in trp for w in s); b = collections.Counter((s[i], s[i + 1]) for s in trp for i in range(len(s) - 1)); t3 = collections.Counter((s[i], s[i + 1], s[i + 2]) for s in trp for i in range(len(s) - 2))
bc = collections.Counter(s[i] for s in trp for i in range(len(s) - 1)); N = sum(u.values())
# KN-style continuation counts for interpolated absolute discounting (bigram)
cont = collections.Counter(w2 for (w1, w2) in b); n_types = len(b); fol = collections.Counter(w1 for (w1, w2) in b)
def p_uni(w): return (u[w] + 1) / (N + V)
def p_bi_add1(w1, w2): return (b[(w1, w2)] + 1) / (bc[w1] + V)
def p_bi_int(w1, w2, lam=0.7): return lam * (b[(w1, w2)] / bc[w1] if bc[w1] else 0) + (1 - lam) * p_uni(w2)
D = 0.75
def p_bi_kn(w1, w2):
    pc = cont[w2] / n_types
    if bc[w1] == 0: return pc
    return max(b[(w1, w2)] - D, 0) / bc[w1] + D * fol[w1] / bc[w1] * pc
bigram_ctx = collections.Counter((s[i], s[i + 1]) for s in trp for i in range(len(s) - 2))
def p_tri_int(w1, w2, w3): 
    c12 = b[(w1, w2)]; pt = t3[(w1, w2, w3)] / c12 if c12 else 0; return 0.5 * pt + 0.3 * (b[(w2, w3)] / bc[w2] if bc[w2] else 0) + 0.2 * p_uni(w3)
def ppl(fn, order):
    ll = n = 0
    for s in tep:
        for i in range(max(order - 1, 1), len(s)):
            p = fn(*s[i - order + 1:i + 1]); ll += math.log(p) if p > 0 else -1e9; n += 1
    return math.exp(-ll / n)
print("vocab", V, "test tokens", sum(len(s) - 1 for s in tep))
print("unigram add-1 perplexity:        %.0f" % ppl(p_uni, 1))
print("bigram MLE perplexity:            inf (unseen bigrams get probability 0)  -> zeros:", sum(1 for s in tep for i in range(len(s) - 1) if b[(s[i], s[i + 1])] == 0), "of", sum(len(s) - 1 for s in tep), "test bigrams unseen")
print("bigram add-1 perplexity:         %.0f" % ppl(p_bi_add1, 2))
print("bigram interpolated (0.7/0.3):   %.0f" % ppl(p_bi_int, 2))
print("bigram Kneser-Ney (D=0.75):      %.0f" % ppl(p_bi_kn, 2))
print("trigram interpolated:            %.0f" % ppl(p_tri_int, 3))

## 3. Word embeddings: PPMI+SVD and skip-gram negative sampling from scratch (chapter N.5)
SGNS takes ~2-3 min on CPU.

In [ ]:
"""Word embeddings (executed): co-occurrence/PPMI+SVD, skip-gram negative sampling from scratch, neighbor checks, WordNet-proxy evaluation."""
import collections, numpy as np, time, warnings; warnings.filterwarnings("ignore")
from nltk.corpus import brown, wordnet as wn
from scipy.stats import spearmanr
from scipy.sparse import coo_matrix
from sklearn.utils.extmath import randomized_svd
rng = np.random.default_rng(0)
toks = [w.lower() for w in brown.words() if w.isalpha()]
cnt = collections.Counter(toks); vocab = [w for w, c in cnt.most_common() if c >= 15]; idx = {w: i for i, w in enumerate(vocab)}; V = len(vocab)
ids = np.array([idx[w] for w in toks if w in idx]); print("tokens", len(toks), "kept", len(ids), "vocab", V)
def neighbors(E, w, k=6):
    En = E / (np.linalg.norm(E, axis=1, keepdims=True) + 1e-9); s = En @ En[idx[w]]; return [vocab[i] for i in np.argsort(-s)[1:k + 1]]

# ---- PPMI + SVD (Levy & Goldberg 2014 view of SGNS) ----
W = 4; rows, cols, vals = [], [], []
for off in range(1, W + 1):
    a, b = ids[:-off], ids[off:]; w = 1.0 / off
    rows += [a, b]; cols += [b, a]; vals += [np.full(len(a), w)] * 2
M = coo_matrix((np.concatenate(vals), (np.concatenate(rows), np.concatenate(cols))), shape=(V, V)).tocsr(); M.sum_duplicates()
tot = M.sum(); rs = np.asarray(M.sum(1)).ravel(); cs = np.asarray(M.sum(0)).ravel() ** 0.75; cs = cs / cs.sum() * tot   # context distribution smoothing alpha=0.75
Mc = M.tocoo(); pmi = np.log((Mc.data * tot) / (rs[Mc.row] * cs[Mc.col])); keep = pmi > 0
P = coo_matrix((pmi[keep], (Mc.row[keep], Mc.col[keep])), shape=(V, V)).tocsr()
U, S, Vt = randomized_svd(P, 100, random_state=0); E_ppmi = U * np.sqrt(S)
print("\nPPMI+SVD(100) neighbors:")
for w in ["president", "car", "church", "money", "war", "university"]: print(f"  {w:10s} -> {neighbors(E_ppmi, w)}")

# ---- skip-gram negative sampling from scratch ----
def sgns(dim=60, epochs=3, neg=5, lr0=0.05, win=4, batch=4096, seed=0):
    r = np.random.default_rng(seed); Win = (r.random((V, dim)) - .5) / dim; Wout = np.zeros((V, dim))
    f = np.array([cnt[w] for w in vocab], float); keep_p = np.minimum(1, np.sqrt(1e-4 / (f / f.sum())) + 1e-4 / (f / f.sum()))   # frequent-word subsampling
    noise = f ** .75; noise /= noise.sum()
    for ep in range(epochs):
        sub = ids[r.random(len(ids)) < keep_p[ids]]; pairs = []
        for off in range(1, win + 1): pairs.append(np.c_[sub[:-off], sub[off:]]); pairs.append(np.c_[sub[off:], sub[:-off]])
        pairs = np.concatenate(pairs); pairs = pairs[r.permutation(len(pairs))]; lr = lr0 * (1 - ep / epochs)
        for s in range(0, len(pairs), batch):
            c, o = pairs[s:s + batch, 0], pairs[s:s + batch, 1]; n = r.choice(V, size=(len(c), neg), p=noise)
            vc = Win[c]; uo = Wout[o]; un = Wout[n]
            gp = 1 / (1 + np.exp(-(vc * uo).sum(1))) - 1; gn = 1 / (1 + np.exp(-np.einsum("bd,bkd->bk", vc, un)))   # gradients of -log sigma(.)
            dvc = gp[:, None] * uo + np.einsum("bk,bkd->bd", gn, un)
            np.add.at(Win, c, -lr * dvc); np.add.at(Wout, o, -lr * gp[:, None] * vc); np.add.at(Wout, n, -lr * gn[:, :, None] * vc[:, None, :])
    return Win
t = time.time(); E_sg = sgns(); print(f"\nSGNS (dim 60, 3 epochs) trained in {time.time()-t:.0f}s; neighbors:")
for w in ["president", "car", "church", "money", "war", "university"]: print(f"  {w:10s} -> {neighbors(E_sg, w)}")

# ---- proxy evaluation: Spearman between cosine and WordNet path similarity over noun pairs ----
nouns = [w for w in vocab[:3000] if wn.synsets(w, "n")][:600]
pairs = [(nouns[i], nouns[j]) for i, j in rng.integers(0, len(nouns), (4000, 2)) if i != j]
def wn_sim(a, b): return max((x.path_similarity(y) or 0) for x in wn.synsets(a, "n")[:3] for y in wn.synsets(b, "n")[:3])
ref = np.array([wn_sim(a, b) for a, b in pairs])
def cos(E, a, b): ea, eb = E[idx[a]], E[idx[b]]; return ea @ eb / (np.linalg.norm(ea) * np.linalg.norm(eb) + 1e-9)
for name, E in [("random vectors", rng.normal(size=(V, 60))), ("PPMI+SVD(100)", E_ppmi), ("SGNS (scratch)", E_sg)]:
    print(f"{name:16s} Spearman vs WordNet path-sim over {len(pairs)} noun pairs: {spearmanr(ref, [cos(E, a, b) for a, b in pairs])[0]:.3f}")

## 4. Sequence labeling: HMM / CRF POS tagging and CRF NER (chapter N.7)

In [ ]:
"""POS tagging (baseline / HMM-Viterbi / CRF) and NER (CRF) -- executed on NLTK treebank and CoNLL-2002 Spanish."""
import collections, numpy as np, math, warnings; warnings.filterwarnings("ignore")
from nltk.corpus import treebank, conll2002
import sklearn_crfsuite
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

sents = [[(w, t) for w, t in s] for s in treebank.tagged_sents(tagset="universal")]
cut = int(len(sents) * .8); tr, te = sents[:cut], sents[cut:]; print("POS: train sents", len(tr), "test sents", len(te), "tokens", sum(len(s) for s in te))
tags = sorted({t for s in tr for _, t in s}); T = len(tags); ti = {t: i for i, t in enumerate(tags)}
wt = collections.defaultdict(collections.Counter)
for s in tr:
    for w, t in s: wt[w.lower()][t] += 1
common = collections.Counter(t for s in tr for _, t in s).most_common(1)[0][0]
def acc(pred): 
    c = n = 0
    for s, p in zip(te, pred): c += sum(a == b for (_, a), b in zip(s, p)); n += len(s)
    return c / n
known = lambda w: w.lower() in wt
base = [[wt[w.lower()].most_common(1)[0][0] if known(w) else common for w, _ in s] for s in te]
print(f"most-frequent-tag baseline:                   {acc(base):.4f}")
oov = sum(not known(w) for s in te for w, _ in s) / sum(len(s) for s in te); print(f"OOV rate in test: {oov:.3f}")
# HMM with add-k smoothing and a crude suffix/shape-based unknown model
trans = np.ones((T + 1, T)) * .01; emit = collections.defaultdict(lambda: np.zeros(T)); start = T
for s in tr:
    prev = start
    for w, t in s: trans[prev, ti[t]] += 1; emit[w.lower()][ti[t]] += 1; prev = ti[t]
trans /= trans.sum(1, keepdims=True); tagc = np.array([sum(emit[w][i] for w in emit) for i in range(T)])
def suffix_model():
    d = collections.defaultdict(lambda: np.zeros(T))
    for s in tr:
        for w, t in s:
            if wt[w.lower()].total() <= 1: d[w[-3:].lower()][ti[t]] += 1
    return d
suf = suffix_model()
def e_prob(w):
    if known(w): v = emit[w.lower()] + 1e-3; return v / tagc  # P(w|t) ~ count(w,t)/count(t)
    v = suf.get(w[-3:].lower(), np.ones(T)) + 1; v = v / v.sum()
    if w[0].isdigit(): v = np.ones(T) * 1e-3; v[ti["NUM"]] = 1
    elif w[0].isupper(): v = v * np.where(np.isin(tags, ["NOUN"]), 3, 1)
    return v / tagc * 1e-3
def viterbi(words):
    n = len(words); dp = np.full((n, T), -1e18); bp = np.zeros((n, T), int)
    dp[0] = np.log(trans[start]) + np.log(e_prob(words[0]))
    for i in range(1, n):
        sc = dp[i - 1][:, None] + np.log(trans[:T]); bp[i] = sc.argmax(0); dp[i] = sc.max(0) + np.log(e_prob(words[i]))
    path = [int(dp[-1].argmax())]
    for i in range(n - 1, 0, -1): path.append(int(bp[i][path[-1]]))
    return [tags[i] for i in reversed(path)]
hmm = [viterbi([w for w, _ in s]) for s in te]; print(f"HMM (bigram tags) + Viterbi:                  {acc(hmm):.4f}")
def feats(s, i):
    w = s[i][0]; f = {"bias": 1, "w.lower": w.lower(), "suf3": w[-3:], "suf2": w[-2:], "pre2": w[:2], "isupper": w.isupper(), "istitle": w.istitle(), "isdigit": w.isdigit(), "hyphen": "-" in w}
    f.update({"-1:w": s[i - 1][0].lower(), "-1:title": s[i - 1][0].istitle()} if i else {"BOS": True}); f.update({"+1:w": s[i + 1][0].lower()} if i < len(s) - 1 else {"EOS": True}); return f
crf = sklearn_crfsuite.CRF(algorithm="lbfgs", c1=0.05, c2=0.05, max_iterations=100, all_possible_transitions=True)
crf.fit([[feats(s, i) for i in range(len(s))] for s in tr], [[t for _, t in s] for s in tr]); cp = crf.predict([[feats(s, i) for i in range(len(s))] for s in te])
print(f"CRF with word-shape/context features:         {acc(cp):.4f}")
unk = [(i, j) for i, s in enumerate(te) for j, (w, _) in enumerate(s) if not known(w)]
for name, P in [("baseline", base), ("HMM", hmm), ("CRF", cp)]: print(f"  accuracy on OOV tokens ({len(unk)}) -- {name}: {np.mean([P[i][j] == te[i][j][1] for i, j in unk]):.3f}")

print("\n== NER: CoNLL-2002 Spanish, CRF, entity-level F1 ==")
trn = conll2002.iob_sents("esp.train"); tst = conll2002.iob_sents("esp.testb"); print("train sents", len(trn), "test sents", len(tst))
def nf(s, i):
    w, p, _ = s[i]; f = {"bias": 1, "w.lower": w.lower(), "w[-3:]": w[-3:], "w[-2:]": w[-2:], "isupper": w.isupper(), "istitle": w.istitle(), "isdigit": w.isdigit(), "postag": p, "postag2": p[:2]}
    if i: w1, p1, _ = s[i - 1]; f.update({"-1:w.lower": w1.lower(), "-1:istitle": w1.istitle(), "-1:postag": p1})
    else: f["BOS"] = True
    if i < len(s) - 1: w1, p1, _ = s[i + 1]; f.update({"+1:w.lower": w1.lower(), "+1:istitle": w1.istitle(), "+1:postag": p1})
    else: f["EOS"] = True
    return f
X = [[nf(s, i) for i in range(len(s))] for s in trn]; Y = [[l for _, _, l in s] for s in trn]
c2 = sklearn_crfsuite.CRF(algorithm="lbfgs", c1=0.1, c2=0.1, max_iterations=100, all_possible_transitions=True).fit(X, Y)
Xt = [[nf(s, i) for i in range(len(s))] for s in tst]; Yt = [[l for _, _, l in s] for s in tst]; Pt = c2.predict(Xt)
print("token accuracy (inflated by 'O'):", round(np.mean([a == b for y, p in zip(Yt, Pt) for a, b in zip(y, p)]), 4))
print(f"entity-level micro  P {precision_score(Yt, Pt):.3f}  R {recall_score(Yt, Pt):.3f}  F1 {f1_score(Yt, Pt):.3f}")
print(classification_report(Yt, Pt, digits=3))
print("majority-'O' token accuracy:", round(np.mean([l == "O" for y in Yt for l in y]), 4), "-> entity F1 0.0")